# Stage 2 — Leakage-Safe Cross-Validation

Purpose: select the best model/feature-set configuration using 5-fold stratified CV,
with all learned preprocessing (MICE, scaling, label encoding) refit inside each fold.

The held-out final test set (2,417 patients) is recreated here using the exact same
split as `02_preproccessing.ipynb`, but is **never used** in this notebook — it is only
set aside for the single final evaluation after model selection is complete.

In [1]:
import sys, sklearn, xgboost, pandas, numpy, scipy

print("Python:", sys.version)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("scipy:", scipy.__version__)

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
pandas: 3.0.5
numpy: 2.5.2
scikit-learn: 1.9.0
xgboost: 3.4.1
scipy: 1.18.1


In [2]:
import subprocess

with open('../results/environment.txt', 'w') as f:
    f.write(f"Python: {sys.version}\n\n")
    f.write(subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout)

print("Saved environment.txt")

Saved environment.txt


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold

df = pd.read_excel("../data/raw/dataset.xlsx")

binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']
for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

# EXACT same split as 02_preproccessing.ipynb — this recovers the same 9668 training rows and,
# critically, the same 2417 test rows that must stay untouched for the rest of this notebook
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['Disease'], random_state=42
)
df_train_full = df.loc[train_idx].copy()   # CV operates only within this
df_test_final = df.loc[test_idx].copy()    # set aside, do not touch until the very end

print("CV pool:", df_train_full.shape)
print("untouched final test set:", df_test_final.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

CV pool: (9668, 17)
untouched final test set: (2417, 17)


## Fold-safe preprocessing function

Fits every learned step (MICE, scaler, label encoder) on the fold's training portion
only, then transform-only on the fold's validation portion. This mirrors the logic in
`02_preproccessing.ipynb`, refit fresh per fold rather than once globally — necessary
because reusing the already-processed CSVs here would let each fold's validation rows
leak into the shared imputer/scaler that was fit on all 9,668 training rows at once.

In [4]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

mice_cols = ['ESR','CRP','RF','Anti-CCP','HLA-B27','ANA','Anti-Ro','Anti-La','Anti-dsDNA','Anti-Sm','C3','C4']
continuous_cols = ['ESR','CRP','RF','Anti-CCP','C3','C4']
raw_features = ['Age','Gender','ESR','CRP','RF','Anti-CCP','HLA-B27','ANA','Anti-Ro','Anti-La','Anti-dsDNA','Anti-Sm','C3','C4']
ext_features = raw_features + ['RF_was_missing','Anti-CCP_was_missing','Inflammation_Score','C3_C4_Ratio','Autoantibody_Count']

def preprocess_fold(fold_train_df, fold_val_df):
    """
    Fits every learned preprocessing step (MICE, scaler, label encoder) on fold_train ONLY,
    then applies transform-only to fold_val. Nothing here ever calls .fit() on fold_val --
    that's the entire point of doing this per-fold instead of reusing the global CSVs.
    """
    ft, fv = fold_train_df.copy(), fold_val_df.copy()

    # MICE -- fit on fold train, transform both. max_iter=50: verified in 02_preproccessing.ipynb
    # that max_iter=15 does not converge (true convergence is ~iteration 17); 15 produced
    # unstable, non-reproducible downstream results.
    imputer = IterativeImputer(random_state=42, max_iter=50)
    ft[mice_cols] = imputer.fit_transform(ft[mice_cols])
    fv[mice_cols] = imputer.transform(fv[mice_cols])

    for col in ['HLA-B27','ANA','Anti-Ro','Anti-La','Anti-dsDNA','Anti-Sm']:
        ft[col] = ft[col].round().clip(0, 1)
        fv[col] = fv[col].round().clip(0, 1)
    for col in continuous_cols:
        ft[col] = ft[col].clip(lower=0)
        fv[col] = fv[col].clip(lower=0)

    for d in (ft, fv):
        # Inflammation_Score = ESR + CRP: an ad-hoc composite, NOT a validated clinical score.
        # Refer to this in the paper as an "ESR-CRP composite feature," not a clinical "score."
        d['Inflammation_Score'] = d['ESR'] + d['CRP']

        # C3_C4_Ratio: exploratory, hypothesis-driven (complement consumption differs in SLE).
        # .replace(0, pd.NA) guards against a silent division-by-zero -> inf if C4 is ever
        # imputed as slightly negative and clipped to exactly 0 (not observed in current data --
        # verified min C4 = 5.0 in the raw dataset -- but defensive against future/different data).
        d['C3_C4_Ratio'] = d['C3'] / d['C4'].replace(0, pd.NA)

        # Autoantibody_Count: count of positive results among the 6 binary markers.
        # Exploratory aggregate feature, not a validated diagnostic index.
        d['Autoantibody_Count'] = d[['HLA-B27','ANA','Anti-Ro','Anti-La','Anti-dsDNA','Anti-Sm']].sum(axis=1)

    le = LabelEncoder()
    ft['Disease_encoded'] = le.fit_transform(ft['Disease'])
    fv['Disease_encoded'] = le.transform(fv['Disease'])

    X_raw_tr, X_raw_va = ft[raw_features].copy(), fv[raw_features].copy()
    X_ext_tr, X_ext_va = ft[ext_features].copy(), fv[ext_features].copy()
    y_tr, y_va = ft['Disease_encoded'], fv['Disease_encoded']

    # scaler -- fit on fold train, transform both
    scaler_raw = StandardScaler()
    X_raw_tr[continuous_cols+['Age']] = scaler_raw.fit_transform(X_raw_tr[continuous_cols+['Age']])
    X_raw_va[continuous_cols+['Age']] = scaler_raw.transform(X_raw_va[continuous_cols+['Age']])

    ext_continuous = continuous_cols + ['Age', 'Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']
    scaler_ext = StandardScaler()
    X_ext_tr[ext_continuous] = scaler_ext.fit_transform(X_ext_tr[ext_continuous])
    X_ext_va[ext_continuous] = scaler_ext.transform(X_ext_va[ext_continuous])

    return X_raw_tr, X_raw_va, X_ext_tr, X_ext_va, y_tr, y_va

## Fold loop -- trains all 6 model/feature-set combinations per fold

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

cv_results = []  # one row per (fold, model, feature_set) -- raw material for the mean+-std table

fold_num = 0
for train_fold_idx, val_fold_idx in skf.split(df_train_full, df_train_full['Disease']):
    fold_num += 1
    fold_train_df = df_train_full.iloc[train_fold_idx]
    fold_val_df   = df_train_full.iloc[val_fold_idx]

    # this is the one call that does all fold-local fitting (MICE, scaler, label encoder)
    X_raw_tr, X_raw_va, X_ext_tr, X_ext_va, y_tr, y_va = preprocess_fold(fold_train_df, fold_val_df)
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)

    feature_sets = {'raw': (X_raw_tr, X_raw_va), 'ext': (X_ext_tr, X_ext_va)}

    for feat_label, (X_tr, X_va) in feature_sets.items():
        # fresh model instances every time -- no state carried between feature sets or folds
        models = {
            'Logistic Regression': LogisticRegression(solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
            'XGBoost': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                      objective='multi:softprob', eval_metric='mlogloss',
                                      random_state=42, n_jobs=-1),
        }

        for model_name, model in models.items():
            if model_name == 'XGBoost':
                model.fit(X_tr, y_tr, sample_weight=sample_weights)
            else:
                model.fit(X_tr, y_tr)
            y_pred = model.predict(X_va)

            cv_results.append({
                'fold': fold_num,
                'model': model_name,
                'features': feat_label,
                'macro_f1': f1_score(y_va, y_pred, average='macro'),
                'weighted_f1': f1_score(y_va, y_pred, average='weighted'),
                'accuracy': accuracy_score(y_va, y_pred),
                'balanced_accuracy': balanced_accuracy_score(y_va, y_pred),
            })

    print(f"Fold {fold_num} done")

cv_results_df = pd.DataFrame(cv_results)
cv_results_df

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


,fold,model,features,macro_f1,weighted_f1,accuracy,balanced_accuracy
0,1,Logistic Regression,raw,0.779098,0.786442,0.785419,0.806558
1,1,Random Forest,raw,0.807253,0.812937,0.817477,0.814425
2,1,XGBoost,raw,0.804211,0.810815,0.811789,0.808459
3,1,Logistic Regression,ext,0.771378,0.775698,0.776112,0.799062
4,1,Random Forest,ext,0.813047,0.818378,0.821613,0.823483
5,1,XGBoost,ext,0.805286,0.811635,0.812823,0.809568
6,2,Logistic Regression,raw,0.776329,0.788450,0.784902,0.805682
7,2,Random Forest,raw,0.816076,0.829730,0.831954,0.823970
8,2,XGBoost,raw,0.817628,0.830960,0.830920,0.822449
9,2,Logistic Regression,ext,0.783425,0.794255,0.791107,0.812097


In [6]:
summary = cv_results_df.groupby(['model', 'features']).agg(
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
    weighted_f1_mean=('weighted_f1', 'mean'),
    weighted_f1_std=('weighted_f1', 'std'),
    accuracy_mean=('accuracy', 'mean'),
    accuracy_std=('accuracy', 'std'),
    balanced_accuracy_mean=('balanced_accuracy', 'mean'),
    balanced_accuracy_std=('balanced_accuracy', 'std'),
).round(4).sort_values('macro_f1_mean', ascending=False)

summary.to_csv('../results/cv_summary.csv')
summary

macro_f1_mean  macro_f1_std  weighted_f1_mean  \
model               features                                                  
Random Forest       ext              0.8153        0.0016            0.8255   
XGBoost             ext              0.8128        0.0086            0.8224   
Random Forest       raw              0.8106        0.0067            0.8217   
XGBoost             raw              0.8068        0.0074            0.8172   
Logistic Regression ext              0.7860        0.0092            0.7956   
                    raw              0.7845        0.0066            0.7952   

                              weighted_f1_std  accuracy_mean  accuracy_std  \
model               features                                                 
Random Forest       ext                0.0041         0.8283        0.0041   
XGBoost             ext                0.0099         0.8234        0.0101   
Random Forest       raw                0.0075         0.8257        0.0065   
XGBoost             raw                0.0086         0.8182        0.0082   
Logistic Regression ext                0.0120         0.7935        0.0109   
                    raw                0.0079         0.7930        0.0079   

                              balanced_accuracy_mean  balanced_accuracy_std  
model               features                                                 
Random Forest       ext                       0.8267                 0.0039  
XGBoost             ext                       0.8195                 0.0078  
Random Forest       raw                       0.8204                 0.0071  
XGBoost             raw                       0.8151                 0.0060  
Logistic Regression ext                       0.8157                 0.0106  
                    raw                       0.8151                 0.0088